In [6]:
from functions.utils.cloudstorage import GoogleCloudStorage
from functions.utils.bigquery import DataQuery

In [253]:
import json
import numpy as np
import io
from datetime import datetime, timedelta, timezone
from google.cloud import storage

class GoogleCloudStorage:
    def __init__(self,bucket_name):
        self.bucket_name = bucket_name
        self.client = storage.Client()
        self.bucket_exists = False
        try:
            self.bucket = self.client.get_bucket(bucket_name)
            self.bucket_exists = True
            print(f"Bucket exists  : {bucket_name}")
        except:
            self.bucket = None
            print(f"Bucket NOT exists : {bucket_name}")
            
    def blob_exists(self, blob_path) -> bool:
        '''check if object exists'''
        return self.bucket.blob(blob_path).exists()

    ### ---------- Read file function ----------- ###
    def read_json(self, blob_path):
        '''read json file'''
        print("read_json...")
        blob   = self.bucket.blob(blob_path)
        return json.loads(blob.download_as_text())

    def read_text(self, blob_path):
        '''read text file'''
        blob   = self.bucket.blob(blob_path)
        return blob.download_as_text()

    def read_npy(self, blob_path):
        '''read .npy (embedding vector) file'''
        blob   = self.bucket.blob(blob_path)

        buffer = io.BytesIO()
        blob.download_to_file(buffer)
        buffer.seek(0)
        return np.load(buffer)

    def prefix_exists(self,prefix:str)->bool:
        '''Do any blobs start with this profex'''
        if not prefix.endswith("/"):
            prefix = prefix + "/"
        blobs = list(self.bucket.list_blobs(prefix=prefix, max_results=1))
        return len(blobs) > 0 

    def blob_exists(self, blob_path:str) -> bool:
        return self.bucket.blob(blob_path).exists()

    def _build_metadata_from_bigquery(self,student_id:str)->dict:
        print("activate query data from bigquery function ...")
        dq = DataQuery()
        dqs = dq.get_students([student_id])
        if dqs.empty:
            return {}
        row = dqs.iloc[0].to_dict()
        return {
            "student_id": row["student_id"],
            "current_status": row["current_status"],
            "education_level": row["education_level"],
            "education_major": row["education_major"],
            "target_roles": row["target_roles"],
            "timezone": "UTC",
            "model_name": "gemini-2.5-flash",
            "max_output_tokens": 1024,
            "feed_text_max_chars": 872,
            "temperature": 0.1,
        }

    def retrieve_student_bundle(self, student_id,embedding_names):
        result = {
            "metadata":{},
            "embeddings":{},
            "status":""
        }
        ### ---------bucket-------- ###
        if self.bucket_exists:
            result["status"] += f"{self.bucket_name} /\n"
        else:
            result["status"] += f"{self.bucket_name} x\n"
        ### --------- student --------- ###
        student_prefix = f"{student_id}"
        if self.prefix_exists(student_prefix):
            result["status"] += f"|- {student_id} /\n"
        else:
            result["status"] += f"|- {student_id} x\n"

        ### ----------preparepart---------- ###
        metadata_prefix = f"{student_id}/metadata"
        metadata_path   = f"{student_id}/metadata/metadata.json"
        embedding_prefix = f"{student_id}/embedding"
        
        ### ----------metatada---------- ###       
        if self.prefix_exists(metadata_prefix): # if we have blob
            result["status"] += "  |- metadata folder /\n"
            if self.blob_exists(metadata_path):
                result["metadata"] =  self.read_json(metadata_path)
                result["status"] += "    |- metadata.json /\n"
            else:
                result["status"] += "    |- metadata.json x\n"
                # accivate query from BigQuery function
                print(f"activate query data from bigquery function ...")
                result["metadata"] = self._build_metadata_from_bigquery(student_id)
        else:
            result["status"] += "  |- metadata folder x\n"
            result["status"] += "    |- metadata.json x\n"
            print(f"activate query data from bigquery function ...")
            dq = DataQuery()
            dqs = dq.get_students([student_id])
            dqs_dict = dqs.iloc[0].to_dict()
            result["metadata"] = self._build_metadata_from_bigquery(student_id)
            
        ### ----------embedding---------- ###
        print(f"- embedding_prefix -> {embedding_prefix}")
        if self.prefix_exists(embedding_prefix):
            result["status"] += "  |- embedding folder /\n"
            for name in embedding_names:
                path = f"{embedding_prefix}/{name}"
                if self.blob_exists(path):
                    result["status"] += f"    |- {path} /\n"
                    result["embeddings"][name] = self.read_npy(path)[0:3]
                else:
                    result["embeddings"][name] = np.array([])
                    result["status"] += f"    |- {path} x\n"
        else:
            result["status"] += "  |- embedding folder x\n"
            for name in embedding_names:
                path = f"{embedding_prefix}/{name}"
                result["status"] += f"    |- {path} x\n"
                result["embeddings"][name] = np.array([])
                
        return result
        
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake")
student_id = "stu_p001"
x = cgs.retrieve_student_bundle(student_id,
                                ["embedding01.npy", "embedding02.npy", "embedding03.npy","embedding04.npy","embedding05.npy"]
                               )
print()
print(x['status'])

Bucket exists  : hyde-datalake
- metadata_prefix  -> stu_p001/metadata
- metadata_path    -> stu_p001/metadata/metadata.json
activate query data from bigquery function ...


/usr/local/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


activate query data from bigquery function ...


/usr/local/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


- embedding_prefix -> stu_p001/embedding

hyde-datalake /
|- stu_p001 /
  |- metadata folder x
    |- metadata.json x
  |- embedding folder /
    |- stu_p001/embedding/embedding01.npy /
    |- stu_p001/embedding/embedding02.npy /
    |- stu_p001/embedding/embedding03.npy x
    |- stu_p001/embedding/embedding04.npy /
    |- stu_p001/embedding/embedding05.npy x



In [254]:
x["metadata"]

{'student_id': 'stu_p001',
 'current_status': 'student3yr',
 'education_level': 'bachelor',
 'education_major': 'วิทยาการคอมพิวเตอร์',
 'target_roles': 'Data Analyst',
 'timezone': 'UTC',
 'model_name': 'gemini-2.5-flash',
 'max_output_tokens': 1024,
 'feed_text_max_chars': 872,
 'temperature': 0.1}

In [255]:
x["embeddings"]

{'embedding01.npy': array([ 0.01895305, -0.00955014,  0.00599027], dtype=float32),
 'embedding02.npy': array([ 0.00509049, -0.04070203, -0.0031905 ], dtype=float32),
 'embedding03.npy': array([], dtype=float64),
 'embedding04.npy': array([ 0.01695731,  0.01713481, -0.01191018], dtype=float32),
 'embedding05.npy': array([], dtype=float64)}

<hr>

### case 1 : happy

In [53]:
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake")
student_id = "stu_p001"
cgs.retrieve_student_bundle(student_id,'xxx')

Bucket exists  : hyde-datalake


TypeError: Bucket.list_blobs() got multiple values for argument 'max_results'

TypeError: Bucket.list_blobs() got multiple values for argument 'max_results'

<hr>

In [23]:
cgs.read_npy(f"{student_id}/embedding/embedding01.npy")[0:10]

array([ 0.01895305, -0.00955014,  0.00599027, -0.12227094,  0.00105082,
        0.00462399, -0.01424827, -0.00517897, -0.0113328 , -0.01694888],
      dtype=float32)

In [ ]:
### case 2.3 : 
bucket                          /
|- embedding                    x
    |- embedding01.npy          o
    |- embedding02.npy          o
    |- embedding03.npy          o
    |- embedding04.npy          o
    |- embedding05.npy          o
|- metadata                     /
    |- metadata.json            /

In [ ]:
### case 2.4 : 
bucket                          /
|- embedding                    /
    |- embedding01.npy          /
    |- embedding02.npy          /
    |- embedding03.npy          x
    |- embedding04.npy          x
    |- embedding05.npy          x
|- metadata                     /
    |- metadata.json            /

In [ ]:
### case 2.5 : 
bucket                          /
|- embedding                    /
    |- embedding01.npy          x
    |- embedding02.npy          x
    |- embedding03.npy          x
    |- embedding04.npy          x
    |- embedding05.npy          x
|- metadata                     /
    |- metadata.json            /

<hr>

In [ ]:
### case 3 : 
bucket                          x
|- embedding                    o
    |- embedding01.npy          o
    |- embedding02.npy          o
    |- embedding03.npy          o
    |- embedding04.npy          o
    |- embedding05.npy          o
|- metadta                      o
    |- metadata.json            o